In [1]:
import pandas as pd
from get_snirh import Snirh, Parameters

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Initialize the Client

In [2]:
# Initialize the client for the "piezometria" (groundwater) network.
# Networks are discovered live from SNIRH; pass a slug or a uid.
snirh = Snirh("piezometria", verbose=True)

# List every network SNIRH currently publishes
snirh.networks()

2026-07-15 12:58:50,695 - get_snirh.networks - INFO - Discovered 15 networks


,uid,name,slug
0,458192970,ETA,eta
1,920123705,Hidrométrica,hidrometrica
2,920123000,Hidrométrica Algarve,hidrometrica_algarve
3,111307207,Hidrométrica Açores,hidrometrica_acores
4,100406019,Hidrométrica Madeira,hidrometrica_madeira
5,920123704,Meteorológica,meteorologica
6,111307213,Meteorológica Açores,meteorologica_acores
7,100406020,Meteorológica Madeira,meteorologica_madeira
8,100290943,Nascentes,nascentes
9,100290946,Piezometria,piezometria


## 2. Fetch and Filter Stations

We want to find all stations in the Algarve region. We filter by the following river basins (`basin` column):
- RIBEIRAS DO ALGARVE
- GUADIANA
- ARADE

**Note**: `snirh.stations()` fetches the station table live from SNIRH. If SNIRH is unreachable, it falls back to the snapshot bundled with the package (with a warning telling you how stale the snapshot is). You can refresh the bundled snapshot with `snirh.refresh_snapshot()`.

In [3]:
algarve_basins = ['RIBEIRAS DO ALGARVE', 'GUADIANA', 'ARADE']

print("Fetching station list...")
# stations() returns one merged table (uid, code, name, basin, status, ...).
# Keyword filters match canonical columns case-insensitively; basin accepts a list.
stations = snirh.stations(basin=algarve_basins)

print(f"Found {len(stations)} stations in Algarve basins.")
stations.head()

Fetching station list...


2026-07-15 12:58:52,732 - get_snirh.stations - INFO - Discovered 1127 station uids for network 100290946


2026-07-15 12:58:53,045 - get_snirh.stations - INFO - Fetched metadata for 1127 stations of network 100290946


2026-07-15 12:58:53,049 - get_snirh.stations - INFO - Merged stations table has 1127 rows


Found 475 stations in Algarve basins.


,uid,code,name,latitude,longitude,district,municipality,parish,basin,altitude,coord_x,coord_y,aquifer_system,status
0,34441,387/4,Salvador,39.04669,-6.97828,PORTALEGRE,CAMPO MAIOR,SÃO JOÃO BAPTISTA,GUADIANA,190.00,299975.0,231625.0,A11 - ELVAS - CAMPO MAIOR,NaN
1,2077786,387/8,NaN,39.04855,-7.00261,PORTALEGRE,CAMPO MAIOR,-,GUADIANA,233.00,297866.0,231805.0,A11 - ELVAS - CAMPO MAIOR,NaN
2,17288,399/6,NaN,38.94622,-7.25989,PORTALEGRE,ELVAS,SÃO VICENTE E VENTOSA,GUADIANA,382.35,275702.0,220199.0,A5 - ELVAS - VILA BOIM,NaN
3,34459,399/12,Pena Clara,38.94556,-7.26165,PORTALEGRE,ELVAS,SÃO VICENTE E VENTOSA,GUADIANA,380.00,275550.0,220125.0,A5 - ELVAS - VILA BOIM,NaN
4,34563,400/7,NaN,38.95659,-7.07088,PORTALEGRE,CAMPO MAIOR,NOSSA SENHORA DA EXPECTAÇÃO,GUADIANA,200.00,292075.0,221525.0,A11 - ELVAS - CAMPO MAIOR,NaN


## 3. Station Identifiers

Each station has a human-readable `code` (e.g. `610/183`) and an internal SNIRH `uid`. The data endpoint is queried by `uid`, but you rarely need to handle it yourself: pass the stations DataFrame (or any subset of it) straight to `snirh.timeseries()`, which uses `uid` for the request and `code` to label the results.

In [4]:
# The stations DataFrame carries both identifiers
print(f"First 5 stations:\n{stations[['code', 'uid', 'name', 'basin', 'status']].head()}")

First 5 stations:
     code      uid        name     basin status
0   387/4    34441    Salvador  GUADIANA    NaN
1   387/8  2077786         NaN  GUADIANA    NaN
2   399/6    17288         NaN  GUADIANA    NaN
3  399/12    34459  Pena Clara  GUADIANA    NaN
4   400/7    34563         NaN  GUADIANA    NaN


## 4. Fetch Time-Series Data

We will fetch the "Depth to Groundwater Level" (`GWL_DEPTH`) for these stations. Dates are ISO `YYYY-MM-DD` strings.

To keep this example fast — and polite to the SNIRH servers — we restrict the fetch to five active (`EM SERVIÇO`) stations in the RIBEIRAS DO ALGARVE basin, over 2023–2024. Drop the subsetting and widen the dates to fetch everything.

In [5]:
# Common parameters ship as a convenience enum:
print(Parameters._member_names_)

# You can also discover live which parameters a station actually has data for:
station = stations[stations['status'] == 'EM SERVIÇO'].iloc[0]
print(f"\nParameters with data at station {station['code']} ({station['name']}):")
snirh.parameters(station)

['WIND_DIRECTION_HOURLY', 'EVAPORATION_PICHE_DAILY', 'EVAPORATION_PICHE_MONTHLY', 'EVAPORATION_PAN_DAILY', 'EVAPORATION_PAN_MONTHLY', 'HUMIDITY_RELATIVE_HOURLY', 'HUMIDITY_RELATIVE_AVG_DAILY', 'CLOUD_COVER_DAILY', 'PAN_LEVEL_HOURLY', 'PRECIPITATION_ANNUAL', 'PRECIPITATION_DAILY', 'PRECIPITATION_DAILY_MAX_ANNUAL', 'PRECIPITATION_HOURLY', 'PRECIPITATION_MONTHLY', 'RADIATION_DAILY', 'RADIATION_HOURLY', 'AIR_TEMP_HOURLY', 'AIR_TEMP_MAX_DAILY', 'AIR_TEMP_AVG_DAILY', 'AIR_TEMP_AVG_MONTHLY', 'AIR_TEMP_MIN_DAILY', 'WIND_SPEED_DAILY', 'WIND_SPEED_HOURLY', 'WIND_SPEED_INSTANT', 'WIND_SPEED_MAX_HOURLY', 'WIND_SPEED_AVG_DAILY', 'GWL_DEPTH', 'PIEZOMETRIC_LEVEL']

Parameters with data at station 523/29 (nan):


2026-07-15 12:58:53,852 - get_snirh.parameters - INFO - Discovered 2 parameters for 1 station uid(s)


,uid,name
0,100290981,Nível piezométrico
1,2277,Profundidade Nível Água


In [6]:
print("Fetching GWL data (this may take a moment)...")

# Keep the example polite: five active stations, two years of data.
stations_subset = stations[
    (stations['status'] == 'EM SERVIÇO')
    & (stations['basin'] == 'RIBEIRAS DO ALGARVE')
].head(5)

# Pass the DataFrame directly: 'uid' drives the request, 'code' labels the rows.
df_gwl = snirh.timeseries(
    stations_subset,
    Parameters.GWL_DEPTH,
    start='2023-01-01',
    end='2024-12-31',
)

print("Data fetch complete.")

2026-07-15 12:58:53,858 - get_snirh.timeseries - INFO - Fetching timeseries for 5 stations, parameter GWL_DEPTH (2277), 01/01/2023..31/12/2024, 5 workers


Fetching GWL data (this may take a moment)...


2026-07-15 12:58:54,231 - get_snirh.timeseries - INFO - Fetched 94 rows total


Data fetch complete.


## 5. Explore the Data

In [7]:
print(f"Total records fetched: {len(df_gwl)}")
df_gwl.head()

Total records fetched: 94


,timestamp,code,uid,parameter,value
0,2023-01-18,576/1,100094,GWL_DEPTH,2.06
1,2023-02-08,576/1,100094,GWL_DEPTH,1.98
2,2023-03-06,576/1,100094,GWL_DEPTH,1.81
3,2023-04-01,576/1,100094,GWL_DEPTH,1.81
4,2023-05-08,576/1,100094,GWL_DEPTH,2.06


In [8]:
# Check unique stations with data
stations_with_data = df_gwl['code'].nunique()
print(f"Stations with data returned: {stations_with_data} of {len(stations_subset)} requested")

Stations with data returned: 5 of 5 requested


## 6. Save to CSV

In [9]:
output_file = 'algarve_gwl_data.csv'
df_gwl.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")

Data saved to algarve_gwl_data.csv


## 7. Other Networks: Meteorology

The same facade works for any SNIRH network — bind a new client to a different network slug (see `snirh.networks()` above for the full list).

In [10]:
# For meteorology
snirh_meteo = Snirh("meteorologica")

In [11]:
algarve_basins = ['RIBEIRAS DO ALGARVE', 'GUADIANA', 'ARADE']

print("Fetching station list...")
stations_meteo = snirh_meteo.stations(basin=algarve_basins)

print(f"Found {len(stations_meteo)} stations in Algarve basins.")
stations_meteo.head()

Fetching station list...


2026-07-15 12:58:55,147 - get_snirh.networks - INFO - Discovered 15 networks


2026-07-15 12:58:57,406 - get_snirh.stations - INFO - Discovered 789 station uids for network 920123704


2026-07-15 12:58:57,918 - get_snirh.stations - INFO - Fetched metadata for 790 stations of network 920123704


2026-07-15 12:58:57,920 - get_snirh.stations - INFO - Merged stations table has 789 rows


Found 133 stations in Algarve basins.


,uid,code,name,latitude,longitude,altitude,latitude_n,longitude_w,coord_x,coord_y,altitude_m_1,basin,district,municipality,parish,entidade_responsavel_automatica,entidade_responsavel_convencional,tipo_estacao_automatica,tipo_estacao_convencional,entrada_funcionamento_convencional,encerramento_convencional,entrada_funcionamento_automatica,encerramento_automatica,telemetria,status,indice_qualidade
0,920684970,21M/02UG,ALANDROAL,38.69300,-7.40400,302.0,38.693,-7.404,263420.797,191939.393,302,GUADIANA,ÉVORA,ALANDROAL,ALANDROAL (NOSSA SENHORA DA CONCEIÇÃO),Autoridade Nacional da Água,CCDR-ALENTEJO,Udográfica,Udométrica,07-09-1931,-,14-02-2001,-,SIM,ATIVA,-
1,920684986,26J/04UG,ALBERNOA,37.85700,-7.96200,133.0,37.857,-7.962,215040.615,98919.839,133,GUADIANA,BEJA,BEJA,ALBERNOA,Autoridade Nacional da Água,CCDR-ALENTEJO,Udográfica,Udométrica,01-05-1979,-,07-02-2001,-,NÃO,ATIVA,-
2,920684976,30E/03F,ALBUFEIRA DA BRAVURA,37.20400,-8.70000,75.0,37.204,-8.7,149671.000,26586.000,75,RIBEIRAS DO ALGARVE,FARO,LAGOS,BENSAFRIM,Autoridade Nacional da Água,-,Climatológica Flutuante,-,-,-,04-04-2001,22-10-2017,NÃO,DESATIVADA,-
3,920684980,30M/05F,ALBUFEIRA DE ODELEITE,37.32700,-7.48800,41.0,37.327,-7.488,257141.000,40280.000,41,GUADIANA,FARO,CASTRO MARIM,ODELEITE,Autoridade Nacional da Água,-,Climatológica Flutuante,-,-,-,06-04-2001,28-03-2018,NÃO,DESATIVADA,-
4,920684974,24L/02F,ALBUFEIRA DO ALQUEVA,38.22317,-7.45971,79.0,38.22317,-7.45971,258965.000,139786.000,79,GUADIANA,BEJA,MOURA,MOURA (SÃO JOÃO BAPTISTA),Autoridade Nacional da Água,-,Climatológica Flutuante,-,-,-,19-07-2002,20-06-2018,NÃO,DESATIVADA,-


In [12]:
print("Fetching annual precipitation data (this may take a moment)...")

# Same politeness limits: a handful of active stations, two years.
meteo_subset = stations_meteo[stations_meteo['status'] == 'ATIVA'].head(5)

df_ppyr = snirh_meteo.timeseries(
    meteo_subset,
    Parameters.PRECIPITATION_ANNUAL,
    start='2023-01-01',
    end='2024-12-31',
)

print("Data fetch complete.")

2026-07-15 12:58:57,928 - get_snirh.timeseries - INFO - Fetching timeseries for 5 stations, parameter PRECIPITATION_ANNUAL (4237), 01/01/2023..31/12/2024, 5 workers


Fetching annual precipitation data (this may take a moment)...


2026-07-15 12:58:58,320 - get_snirh.timeseries - INFO - Fetched 2 rows total


Data fetch complete.


In [13]:
df_ppyr.head()

,timestamp,code,uid,parameter,value
0,2023-10-01 09:00:00,21M/02UG,920684970,PRECIPITATION_ANNUAL,575.7
1,2023-10-01 09:00:00,28J/01G,920684996,PRECIPITATION_ANNUAL,349.8
